# 🦀 Day 7: Mini-Project — CLI Word Counter (`wc`)

Let's build a real CLI tool that combines everything from Month 1!

Our `rwc` (Rust Word Count) tool will:
- Count lines, words, characters, and bytes
- Process multiple inputs
- Format output like Unix `wc`

---

## 📐 Step 1: Core Data Structures

In [ ]:
use std::fs;
use std::io::Write;

#[derive(Debug, Clone)]
struct FileStats {
    name: String,
    lines: usize,
    words: usize,
    chars: usize,
    bytes: usize,
}

impl FileStats {
    fn new(name: &str) -> Self {
        FileStats {
            name: name.to_string(),
            lines: 0, words: 0, chars: 0, bytes: 0,
        }
    }

    fn from_text(name: &str, text: &str) -> Self {
        FileStats {
            name: name.to_string(),
            lines: text.lines().count(),
            words: text.split_whitespace().count(),
            chars: text.chars().count(),
            bytes: text.len(),
        }
    }

    fn add(&self, other: &FileStats) -> FileStats {
        FileStats {
            name: "total".to_string(),
            lines: self.lines + other.lines,
            words: self.words + other.words,
            chars: self.chars + other.chars,
            bytes: self.bytes + other.bytes,
        }
    }
}

println!("FileStats defined! ✅");

## 🖥️ Step 2: Display Engine

In [ ]:
#[derive(Debug, Clone)]
struct DisplayOptions {
    show_lines: bool,
    show_words: bool,
    show_chars: bool,
    show_bytes: bool,
}

impl DisplayOptions {
    fn all() -> Self {
        DisplayOptions {
            show_lines: true, show_words: true,
            show_chars: true, show_bytes: true,
        }
    }

    fn default() -> Self {
        DisplayOptions {
            show_lines: true, show_words: true,
            show_chars: false, show_bytes: true,
        }
    }
}

fn display_stats(stats: &FileStats, opts: &DisplayOptions) {
    let mut parts: Vec<String> = Vec::new();

    if opts.show_lines { parts.push(format!("{:>8}", stats.lines)); }
    if opts.show_words { parts.push(format!("{:>8}", stats.words)); }
    if opts.show_chars { parts.push(format!("{:>8}", stats.chars)); }
    if opts.show_bytes { parts.push(format!("{:>8}", stats.bytes)); }

    println!("{} {}", parts.join(""), stats.name);
}

fn display_header(opts: &DisplayOptions) {
    let mut headers: Vec<String> = Vec::new();
    if opts.show_lines { headers.push(format!("{:>8}", "lines")); }
    if opts.show_words { headers.push(format!("{:>8}", "words")); }
    if opts.show_chars { headers.push(format!("{:>8}", "chars")); }
    if opts.show_bytes { headers.push(format!("{:>8}", "bytes")); }
    println!("{} {}", headers.join(""), "filename");
    println!("{}", "-".repeat(headers.len() * 8 + 10));
}

println!("Display engine ready! ✅");

## ⚙️ Step 3: File Processing

In [ ]:
fn process_file(path: &str) -> Result<FileStats, String> {
    let content = fs::read_to_string(path)
        .map_err(|e| format!("rwc: {}: {}", path, e))?;
    Ok(FileStats::from_text(path, &content))
}

fn process_multiple(paths: &[&str]) -> Vec<Result<FileStats, String>> {
    paths.iter().map(|p| process_file(p)).collect()
}

println!("File processor ready! ✅");

## 🚀 Step 4: Run the Word Counter!

In [ ]:
// Create some sample files to process
fs::write("poem.txt",
    "Roses are red,\nViolets are blue,\nRust is fast,\nAnd safe for you."
).unwrap();

fs::write("code.txt",
    "fn main() {\n    println!(\"Hello, world!\");\n    let x = 42;\n    let y = x * 2;\n    println!(\"y = {}\", y);\n}"
).unwrap();

fs::write("data.csv",
    "name,age,city\nAlice,30,Paris\nBob,25,London\nCharlie,35,Tokyo\nDiana,28,Berlin\nEve,32,Sydney"
).unwrap();

println!("Sample files created! ✅");

In [ ]:
// Process files
let files = ["poem.txt", "code.txt", "data.csv"];
let opts = DisplayOptions::all();

println!("\n🧮 rwc — Rust Word Counter\n");
display_header(&opts);

let mut total = FileStats::new("total");
let mut file_count = 0;

for path in &files {
    match process_file(path) {
        Ok(stats) => {
            display_stats(&stats, &opts);
            total = total.add(&stats);
            file_count += 1;
        }
        Err(e) => eprintln!("{}", e),
    }
}

if file_count > 1 {
    println!("{}", "-".repeat(42));
    display_stats(&total, &opts);
}

In [ ]:
// Default mode (lines, words, bytes — like real wc)
println!("\n📋 Default mode (like `wc`):\n");
let default_opts = DisplayOptions::default();
display_header(&default_opts);

for path in &files {
    if let Ok(stats) = process_file(path) {
        display_stats(&stats, &default_opts);
    }
}

In [ ]:
// Error handling
println!("\n⚠️ Error handling:");
match process_file("nonexistent.txt") {
    Ok(stats) => display_stats(&stats, &DisplayOptions::default()),
    Err(e) => println!("{}", e),
}

## 📋 Bonus: Longest Line & Most Common Word

In [ ]:
use std::collections::HashMap;

fn analyze_text(text: &str) {
    // Longest line
    if let Some(longest) = text.lines().max_by_key(|l| l.len()) {
        println!("  Longest line ({} chars): \"{}\"", longest.len(), longest);
    }

    // Average word length
    let words: Vec<&str> = text.split_whitespace().collect();
    if !words.is_empty() {
        let avg_len: f64 = words.iter().map(|w| w.len() as f64).sum::<f64>() / words.len() as f64;
        println!("  Average word length: {:.1} chars", avg_len);
    }

    // Most common word
    let mut freq: HashMap<String, usize> = HashMap::new();
    for word in text.split_whitespace() {
        let clean = word.to_lowercase().trim_matches(|c: char| !c.is_alphanumeric()).to_string();
        if !clean.is_empty() {
            *freq.entry(clean).or_insert(0) += 1;
        }
    }
    if let Some((word, count)) = freq.iter().max_by_key(|(_, &v)| v) {
        println!("  Most common word: '{}' ({} times)", word, count);
    }
}

for path in &files {
    if let Ok(content) = fs::read_to_string(path) {
        println!("\n📊 Analysis of {}:", path);
        analyze_text(&content);
    }
}

// Clean up
for path in &files {
    fs::remove_file(path).ok();
}

## 🎯 Build the Full CLI Tool!

Now take this code and turn it into a real CLI project:

```bash
cargo new rwc
cd rwc
cargo add clap --features derive
```

Structure:
```
rwc/
├── Cargo.toml
└── src/
    ├── main.rs      # CLI entry point with clap
    ├── stats.rs      # FileStats struct and methods
    └── display.rs    # Output formatting
```

Usage:
```bash
$ rwc file1.txt file2.txt
$ rwc -l file.txt           # Lines only
$ rwc -w file.txt           # Words only
$ rwc --all *.txt           # All stats
```

## 🏋️ Extension Challenges

In [ ]:
// Challenge 1: Add a function that finds the top N most frequent words

// YOUR CODE HERE


In [ ]:
// Challenge 2: Add a function that computes reading time
// (assuming 200 words per minute)

// YOUR CODE HERE


In [ ]:
// Challenge 3: Add a function that detects the file encoding
// and handles both ASCII and UTF-8 files

// YOUR CODE HERE


## 🎉 Month 1 Complete!

### What You've Learned:

**Week 1 — Foundations:**
- ✅ Hello World, `println!`, formatting
- ✅ Variables, mutability, constants, shadowing
- ✅ Data types (integers, floats, booleans, chars)
- ✅ Strings (`String` vs `&str`)
- ✅ Tuples and arrays
- ✅ Functions, expressions, return values

**Week 2 — Control Flow & Collections:**
- ✅ `if`/`else`, conditional expressions
- ✅ Loops (`loop`, `while`, `for`)
- ✅ `match` and pattern matching
- ✅ Vectors, HashMaps, iterators

**Week 3 — Structs, Enums & Error Handling:**
- ✅ Structs, methods, associated functions
- ✅ Enums with data variants
- ✅ `Option<T>` and `Result<T, E>`

**Week 4 — Modules, Crates & Cargo:**
- ✅ Module system and visibility
- ✅ External crates and crates.io
- ✅ Cargo: build, test, doc, clippy
- ✅ File I/O and CLI tools

---

### 🚀 Next Month: Ownership & Core Concepts
Get ready for Rust's most unique and powerful features:
ownership, borrowing, lifetimes, generics, and traits!

**Keep committing daily! 💪🦀**